<a href="https://colab.research.google.com/github/murtaza2k/AI_Residency/blob/main/pinecone_rag_explained_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🚀 RAG with Pinecone + LlamaIndex (Deep Explained Version)

This notebook provides **step-by-step explanation + code** to build a RAG system using:

- Pinecone (Vector DB)
- LlamaIndex (RAG framework)
- OpenAI (LLM)

---


## 📦 Step 1: Install Dependencies

In [ ]:
!pip install llama-index pinecone-client openai


## 📚 Step 2: Import Libraries (Explained)

### Core Libraries
- `openai` → connects to LLM (embeddings + responses)  
- `google.colab.userdata` → securely fetch API keys in Colab  

### Pinecone
- `Pinecone` → main client to interact with vector DB  
- `ServerlessSpec` → defines cloud + region  

### LlamaIndex
- `SimpleDirectoryReader` → loads documents  
- `VectorStoreIndex` → builds embedding index  
- `StorageContext` → connects storage layer  
- `PineconeVectorStore` → connects Pinecone to LlamaIndex  

---


In [ ]:

from pinecone import Pinecone, ServerlessSpec
import openai
from google.colab import userdata

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
)
from llama_index.vector_stores.pinecone import PineconeVectorStore



## 🔑 Step 3: Load API Keys

We use Colab secrets:

- `openai` → OpenAI API key  
- `PINECONE_API_KEY` → Pinecone API key  

💡 This avoids hardcoding sensitive credentials.


In [ ]:

openai.api_key = userdata.get('openai')
pinecone_api_key = userdata.get('PINECONE_API_KEY')

print("✅ API Keys Loaded")



## 🌲 Step 4: Create Pinecone Index (Vector Database)

Pinecone stores embeddings for fast similarity search.

### Key Parameters:

- `name="quickstart"` → index name  
- `dimension=1536` → must match OpenAI embedding size  
- `metric="euclidean"` → distance calculation  
- `ServerlessSpec` → defines cloud + region  

### Flow:
1. Connect to Pinecone  
2. Delete old index (optional)  
3. Create new index  
4. Connect to index  

---


In [ ]:

pc = Pinecone(api_key=pinecone_api_key)

# Delete index (optional cleanup)
try:
    pc.delete_index("quickstart")
    print("Old index deleted")
except:
    print("No existing index")

# Create new index
pc.create_index(
    name="quickstart",
    dimension=1536,
    metric="euclidean",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

pinecone_index = pc.Index("quickstart")

print("✅ Pinecone index ready")



## 📂 Step 5: Load Documents

We download a sample document and load it.

### What happens:
- Create folder  
- Download text file  
- Load using LlamaIndex  

---


In [ ]:

!mkdir -p 'data/paul_graham/'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham/paul_graham_essay.txt'


In [ ]:

documents = SimpleDirectoryReader("./data/paul_graham").load_data()
print(f"Loaded {len(documents)} document(s)")



## 🧠 Step 6: Connect LlamaIndex to Pinecone

This is where RAG storage is configured.

### Flow:
1. Create `PineconeVectorStore`
2. Pass it into `StorageContext`
3. Build index → embeddings stored in Pinecone

💡 Now your data is searchable!

---


In [ ]:

vector_store = PineconeVectorStore(pinecone_index=pinecone_index)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)

print("✅ Documents embedded and stored in Pinecone")



## 🔍 Step 7: Query (RAG Pipeline)

### What happens internally:

1. User asks a question  
2. Question → embedding  
3. Pinecone searches similar vectors  
4. Top chunks retrieved  
5. Sent to LLM  
6. Final answer generated  

---


In [ ]:

from IPython.display import Markdown, display

query_engine = index.as_query_engine()

response = query_engine.query("What is this document about?")

display(Markdown(f"**{response}**"))



## 🎯 Final Summary

You built:

✅ Pinecone vector database  
✅ Document embedding pipeline  
✅ LlamaIndex integration  
✅ Full RAG query system  

---

## 🚀 Next Steps

- Add multiple documents  
- Use metadata filtering  
- Build chatbot UI  
- Deploy as API  

---

Happy Building 🚀
